In [ ]:
import json
import os
import sys
import time

REPO_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (REPO_ROOT, os.path.join(REPO_ROOT, "GUI")):
    if path not in sys.path:
        sys.path.insert(0, path)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend
from audio_lab_pynq.hdmi_effect_state_mirror import HdmiEffectStateMirror
from pynq_multi_fx_gui import (
    AppState,
    THEMES,
    make_pynq_static_render_cache,
    render_frame_800x480,
)

HOLD_SECONDS_PER_STEP = 2.0
FINAL_HOLD_SECONDS = 30
RETURN_TO_SAFE_BYPASS = False
THEME = "pipboy-green" if "pipboy-green" in THEMES else None
VARIANT = "compact-v2"
OFFSET_X = 0
OFFSET_Y = 0
RUN_SEQUENCE_ON_START = True

print("Loading AudioLabOverlay() once")
_overlay = AudioLabOverlay()
print("Creating HDMI backend once")
_backend = AudioLabHdmiBackend(_overlay)
_state = AppState()
_cache = make_pynq_static_render_cache()
mirror = HdmiEffectStateMirror(
    overlay=_overlay,
    hdmi_backend=_backend,
    app_state=_state,
    renderer=render_frame_800x480,
    render_cache=_cache,
    theme=THEME,
    variant=VARIANT,
    placement="manual",
    offset_x=OFFSET_X,
    offset_y=OFFSET_Y,
)

class FxControls(object):
    def __init__(self, mirror):
        self.mirror = mirror

    def safe_bypass(self):
        return self.mirror.safe_bypass()

    def basic_clean(self):
        return self.mirror.apply_chain_preset("Basic Clean")

    def noise_gate(self, enabled=True, threshold=25, decay=84, damp=85):
        return self.mirror.set_noise_suppressor_settings(
            enabled=enabled, threshold=threshold, decay=decay, damp=damp)

    def comp(self, enabled=True, threshold=45, ratio=35, response=45, makeup=50):
        return self.mirror.set_compressor_settings(
            enabled=enabled, threshold=threshold, ratio=ratio,
            response=response, makeup=makeup)

    def od(self, enabled=True, drive=35, tone=55, level=65):
        return self.mirror.set_guitar_effects(
            overdrive_on=enabled, overdrive_drive=drive,
            overdrive_tone=tone, overdrive_level=level)

    def dist(self, model="tube_screamer", drive=52, tone=60, level=30,
             bias=50, tight=60, mix=100):
        return self.mirror.set_pedal_model(
            model, drive=drive, tone=tone, level=level,
            bias=bias, tight=tight, mix=mix)

    def clean_boost(self, drive=30, tone=None, level=60, mix=100):
        return self.mirror.clean_boost(
            drive=drive, tone=tone, level=level, mix=mix)

    def tube_screamer(self, drive=45, tone=55, level=65, mix=100):
        return self.mirror.tube_screamer(
            drive=drive, tone=tone, level=level, mix=mix)

    def rat(self, drive=55, filter=45, level=60, mix=100):
        return self.mirror.rat(
            drive=drive, filter=filter, level=level, mix=mix)

    def ds1(self, drive=50, tone=50, level=55, mix=100):
        return self.mirror.ds1(
            drive=drive, tone=tone, level=level, mix=mix)

    def big_muff(self, drive=60, tone=45, level=55, mix=100):
        return self.mirror.big_muff(
            drive=drive, tone=tone, level=level, mix=mix)

    def fuzz_face(self, drive=70, tone=40, level=60, mix=100):
        return self.mirror.fuzz_face(
            drive=drive, tone=tone, level=level, mix=mix)

    def metal(self, drive=65, tone=55, level=55, mix=100):
        return self.mirror.metal(
            drive=drive, tone=tone, level=level, mix=mix)

    def amp(self, model="british_crunch", gain=60, bass=55, mid=50,
            treble=45, presence=45, resonance=35, master=75):
        return self.mirror.set_amp_model(
            model, gain=gain, bass=bass, mid=mid, treble=treble,
            presence=presence, resonance=resonance, master=master)

    def jc_clean(self, gain=30, bass=55, mid=50, treble=60):
        return self.mirror.jc_clean(
            gain=gain, bass=bass, mid=mid, treble=treble)

    def clean_combo(self, gain=35, bass=55, mid=55, treble=55):
        return self.mirror.clean_combo(
            gain=gain, bass=bass, mid=mid, treble=treble)

    def british_crunch(self, gain=60, bass=50, mid=65, treble=55):
        return self.mirror.british_crunch(
            gain=gain, bass=bass, mid=mid, treble=treble)

    def high_gain_stack(self, gain=70, bass=55, mid=50, treble=60):
        return self.mirror.high_gain_stack(
            gain=gain, bass=bass, mid=mid, treble=treble)

    def cab(self, model="2x12", air=40, mix=100, level=100):
        return self.mirror.cab(
            model=model, air=air, mix=mix, level=level)

    def eq(self, enabled=True, low=50, mid=55, high=60):
        return self.mirror.eq(enabled=enabled, low=low, mid=mid, high=high)

    def reverb(self, enabled=True, mix=25, decay=50, tone=65):
        return self.mirror.reverb(
            enabled=enabled, mix=mix, decay=decay, tone=tone)

    def render(self):
        return self.mirror.render(reason="manual")

    def summary(self):
        data = self.mirror.get_state_summary()
        print(json.dumps(data, indent=2, sort_keys=True, default=str))
        return data

    def selected_history(self):
        self.mirror.print_selected_fx_history()
        return list(self.mirror.selected_fx_history)

fx = FxControls(mirror)

def _snapshot():
    return {
        "pedal": mirror.current_pedal_label,
        "amp": mirror.current_amp_label,
        "cab": mirror.current_cab_label,
    }

def _run_step(step, operation, expected, callback):
    print("[{0:02d}] {1}".format(step, operation))
    try:
        callback()
        mirror.assert_selected_fx(expected)
        actual = mirror.get_selected_fx_actual()
        result = "PASS"
    except Exception as exc:
        actual = mirror.get_selected_fx_actual()
        result = "FAIL: {0!r}".format(exc)
    snap = _snapshot()
    info = mirror.last_render_info or {}
    print("expected SELECTED FX: {0}".format(expected))
    print("actual SELECTED FX  : {0}".format(actual))
    print("pedal model         : {0}".format(snap["pedal"]))
    print("amp model           : {0}".format(snap["amp"]))
    print("cab model           : {0}".format(snap["cab"]))
    print("result              : {0}".format(result))
    print("render/copy         : {0:.3f}s / {1}".format(
        info.get("render_s") or 0.0, info.get("framebuffer_copy_s")))
    print("VDMA error bits     : {0}".format(info.get("hdmi_errors")))
    print("")
    if result != "PASS":
        raise AssertionError(result)
    if HOLD_SECONDS_PER_STEP > 0:
        time.sleep(HOLD_SECONDS_PER_STEP)

print("Initial 800x480 HDMI GUI render at x=0 y=0")
mirror.render(reason="initial")

if RUN_SEQUENCE_ON_START:
    _sequence = [
        ("fx.clean_boost", "CLEAN BOOST", fx.clean_boost),
        ("fx.tube_screamer", "TUBE SCREAMER", fx.tube_screamer),
        ("fx.rat", "RAT", fx.rat),
        ("fx.ds1", "DS-1", fx.ds1),
        ("fx.big_muff", "BIG MUFF", fx.big_muff),
        ("fx.fuzz_face", "FUZZ FACE", fx.fuzz_face),
        ("fx.metal", "METAL", fx.metal),
        ("fx.jc_clean", "AMP SIM", fx.jc_clean),
        ("fx.british_crunch", "AMP SIM", fx.british_crunch),
        ("fx.high_gain_stack", "AMP SIM", fx.high_gain_stack),
        ("fx.cab model=2x12", "CAB", fx.cab),
        ("fx.reverb", "REVERB", fx.reverb),
    ]
    for _idx, (_operation, _expected, _callback) in enumerate(_sequence, 1):
        _run_step(_idx, _operation, _expected, _callback)

if RETURN_TO_SAFE_BYPASS:
    fx.safe_bypass()

fx.selected_history()
print("User operations:")
print("  fx.clean_boost(drive=30, level=60)")
print("  fx.tube_screamer(drive=45, tone=55, level=65)")
print("  fx.rat(drive=55, filter=45, level=60)")
print("  fx.ds1(drive=50, tone=50, level=55)")
print("  fx.big_muff(drive=60, tone=45, level=55)")
print("  fx.fuzz_face(drive=70, tone=40, level=60)")
print("  fx.metal(drive=65, tone=55, level=55)")
print("  fx.jc_clean(gain=30, bass=55, mid=50, treble=60)")
print("  fx.clean_combo(gain=35, bass=55, mid=55, treble=55)")
print("  fx.british_crunch(gain=60, bass=50, mid=65, treble=55)")
print("  fx.high_gain_stack(gain=70, bass=55, mid=50, treble=60)")
print("  fx.cab(model=\"2x12\", air=40)")
print("  fx.reverb(mix=25, decay=50)")
print("  fx.safe_bypass()")
print("fx, mirror, and _state are ready")

if FINAL_HOLD_SECONDS > 0:
    print("Final HDMI hold: {0} seconds".format(FINAL_HOLD_SECONDS))
    time.sleep(FINAL_HOLD_SECONDS)
